# 🚚 Logistics Operations Analytics — Power BI Dataset
### End-to-End Data Analysis | KPI Dashboard | Business Insights
---
**Dataset:** Indian Logistics Company — 300 Orders, 15 Customers, 10 Products, 5 Warehouses, 12 Drivers

**Objectives:**
- 📦 Explore orders, revenue, profit and delivery performance
- 📊 Answer all suggested KPIs from the dataset
- 🔍 Identify delayed orders, unprofitable products, and top performers
- 📈 Build visual insights ready for Power BI / GitHub portfolio


## 0. Imports & Setup

In [ ]:
import warnings, os
warnings.filterwarnings('ignore')

import numpy  as np
import pandas as pd
import matplotlib.pyplot  as plt
import matplotlib.ticker  as mticker
import seaborn            as sns
from   matplotlib.patches import Patch

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (13, 5)})
pd.set_option('display.float_format', '{:,.2f}'.format)

print("✅  Libraries ready.")


## 1. Load All Sheets

In [ ]:
FILE = 'Logistics_PowerBI_Dataset__1_.xlsx'   # update path if needed

xl         = pd.read_excel(FILE, sheet_name=None)
orders     = xl['Orders'].copy()
customers  = xl['Customers'].copy()
products   = xl['Products'].copy()
warehouses = xl['Warehouses'].copy()
drivers    = xl['Drivers'].copy()

# Fix OrderDate
orders['OrderDate'] = pd.to_datetime(orders['OrderDate'], errors='coerce')
orders['Month']     = orders['OrderDate'].dt.to_period('M').astype(str)
orders['MonthNum']  = orders['OrderDate'].dt.month
orders['Quarter']   = orders['OrderDate'].dt.to_period('Q').astype(str)

print(f"Orders : {orders.shape}")
print(f"Customers  : {customers.shape}")
print(f"Products   : {products.shape}")
print(f"Warehouses : {warehouses.shape}")
print(f"Drivers    : {drivers.shape}")
orders.head()


## 2. Data Inspection & Quality Check

In [ ]:
print("="*55)
print("ORDERS — Basic Info")
print("="*55)
orders.info()


In [ ]:
print("\nMissing Values:")
print(orders.isnull().sum())
print("\nDuplicate rows:", orders.duplicated().sum())
print("\nDate range:", orders['OrderDate'].min().date(), "→", orders['OrderDate'].max().date())
print("\nDelivery Status breakdown:")
print(orders['DeliveryStatus'].value_counts())


In [ ]:
print("Numeric Summary:")
orders[['Quantity','Revenue','ShippingCost','TotalCost','Profit','DeliveryDays']].describe().T


## 3. Build Master (Merged) Table

In [ ]:
master = (orders
    .merge(customers,  on='CustomerID',  how='left')
    .merge(products,   on='ProductID',   how='left')
    .merge(warehouses, on='WarehouseID', how='left')
    .merge(drivers,    on='DriverID',    how='left')
)

master.rename(columns={
    'CustomerName':'Customer', 'City_x':'CustomerCity',
    'Region':'CustomerRegion', 'ProductName':'Product',
    'Category':'ProductCategory', 'UnitPrice':'UnitPrice',
    'WarehouseName':'Warehouse', 'City_y':'WarehouseCity',
    'DriverName':'Driver', 'VehicleType':'Vehicle'
}, inplace=True)

print(f"Master table shape: {master.shape}")
master.head(3)


## 4. 📊 Key Performance Indicators (KPIs)

In [ ]:
total_revenue   = master['Revenue'].sum()
total_profit    = master['Profit'].sum()
total_orders    = master['OrderID'].nunique()
avg_del_days    = master['DeliveryDays'].mean()
delayed_orders  = (master['DeliveryStatus'] == 'Delayed').sum()
delay_rate      = delayed_orders / total_orders * 100
profit_margin   = total_profit / total_revenue * 100
avg_order_value = total_revenue / total_orders
neg_profit_orders = (master['Profit'] < 0).sum()

print("=" * 50)
print("     📦  LOGISTICS KPI SUMMARY")
print("=" * 50)
print(f"  Total Revenue       : ₹{total_revenue:>15,.0f}")
print(f"  Total Profit        : ₹{total_profit:>15,.0f}")
print(f"  Profit Margin       :  {profit_margin:>14.1f}%")
print(f"  Total Orders        :  {total_orders:>15,}")
print(f"  Avg Order Value     : ₹{avg_order_value:>15,.0f}")
print(f"  Avg Delivery Days   :  {avg_del_days:>14.1f}")
print(f"  Delayed Orders      :  {delayed_orders:>14,}  ({delay_rate:.1f}%)")
print(f"  Negative-Profit     :  {neg_profit_orders:>14,}")
print("=" * 50)


In [ ]:
# ── KPI Dashboard Visual ──────────────────────────────────────────────────────
kpis = {
    'Total\nRevenue':  f'₹{total_revenue/1e6:.1f}M',
    'Total\nProfit':   f'₹{total_profit/1e6:.1f}M',
    'Profit\nMargin':  f'{profit_margin:.1f}%',
    'Total\nOrders':   f'{total_orders}',
    'Delayed\nOrders': f'{delayed_orders}\n({delay_rate:.0f}%)',
    'Avg Delivery\nDays': f'{avg_del_days:.1f}',
}
colors = ['#2196F3','#4CAF50','#FF9800','#9C27B0','#F44336','#00BCD4']

fig, axes = plt.subplots(1, 6, figsize=(17, 3))
for ax, (k, v), c in zip(axes, kpis.items(), colors):
    ax.set_facecolor(c)
    ax.text(0.5, 0.6, v, ha='center', va='center', fontsize=16,
            fontweight='bold', color='white', transform=ax.transAxes)
    ax.text(0.5, 0.2, k, ha='center', va='center', fontsize=9,
            color='white', alpha=0.9, transform=ax.transAxes)
    ax.axis('off')
fig.suptitle('Logistics KPI Dashboard', fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout(pad=0.3)
plt.savefig('kpi_dashboard.png', bbox_inches='tight')
plt.show()


## 5. Revenue & Profit Analysis

In [ ]:
# ── Monthly Revenue & Profit trend ────────────────────────────────────────────
monthly = (master.groupby('Month')[['Revenue','Profit']]
                 .sum()
                 .reset_index()
                 .sort_values('Month'))

fig, ax = plt.subplots(figsize=(14, 5))
x = range(len(monthly))
ax.bar(x, monthly['Revenue'], color='#2196F3', alpha=0.7, label='Revenue', width=0.4,
       align='center')
ax.bar([i+0.4 for i in x], monthly['Profit'], color='#4CAF50', alpha=0.8,
       label='Profit', width=0.4, align='center')
ax.set_xticks([i+0.2 for i in x])
ax.set_xticklabels(monthly['Month'], rotation=40, ha='right', fontsize=9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'₹{v/1e3:.0f}K'))
ax.set_title('Monthly Revenue vs Profit', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('monthly_revenue_profit.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── Revenue by Region ─────────────────────────────────────────────────────────
region_rev = master.groupby('CustomerRegion')[['Revenue','Profit']].sum().sort_values('Revenue', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors_reg = ['#2196F3','#FF9800','#4CAF50','#9C27B0']

axes[0].bar(region_rev.index, region_rev['Revenue'], color=colors_reg, edgecolor='white')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'₹{v/1e6:.1f}M'))
axes[0].set_title('Revenue by Region', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Region')
axes[0].set_ylabel('Revenue (₹)')

axes[1].bar(region_rev.index, region_rev['Profit'], color=colors_reg, edgecolor='white')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'₹{v/1e6:.1f}M'))
axes[1].set_title('Profit by Region', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Region')

plt.tight_layout()
plt.savefig('region_revenue_profit.png', bbox_inches='tight')
plt.show()
print("\nRegion-wise Revenue & Profit:")
print(region_rev.to_string())


## 6. Product Performance Analysis

In [ ]:
prod = (master.groupby(['Product','ProductCategory'])
              .agg(TotalRevenue=('Revenue','sum'),
                   TotalProfit=('Profit','sum'),
                   TotalQty=('Quantity','sum'),
                   OrderCount=('OrderID','count'))
              .reset_index()
              .sort_values('TotalRevenue', ascending=False))

prod['ProfitMargin%'] = (prod['TotalProfit'] / prod['TotalRevenue'] * 100).round(1)

print("Product Performance Table:")
print(prod[['Product','ProductCategory','TotalRevenue','TotalProfit','TotalQty','ProfitMargin%']].to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
pal = sns.color_palette('tab10', len(prod))

# Revenue by product
bars = axes[0].barh(prod['Product'], prod['TotalRevenue'], color=pal, edgecolor='white')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'₹{v/1e6:.1f}M'))
axes[0].set_title('Total Revenue by Product', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Revenue (₹)')
for bar, val in zip(bars, prod['TotalRevenue']):
    axes[0].text(bar.get_width()*1.01, bar.get_y()+bar.get_height()/2,
                 f'₹{val/1e6:.2f}M', va='center', fontsize=8)

# Profit margin
colors_margin = ['#4CAF50' if x > 0 else '#F44336' for x in prod['ProfitMargin%']]
axes[1].barh(prod['Product'], prod['ProfitMargin%'], color=colors_margin, edgecolor='white')
axes[1].axvline(0, color='black', linewidth=0.8, linestyle='--')
axes[1].set_title('Profit Margin % by Product', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Profit Margin (%)')

plt.tight_layout()
plt.savefig('product_performance.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── Top Selling Product (by Quantity) ─────────────────────────────────────────
top_product = prod.nlargest(1, 'TotalQty')
print(f"🏆 Top Selling Product (Qty): {top_product.iloc[0]['Product']}")
print(f"   Total Units Sold : {top_product.iloc[0]['TotalQty']:,}")
print(f"   Total Revenue    : ₹{top_product.iloc[0]['TotalRevenue']:,.0f}")
print()
print("Full Quantity Ranking:")
print(prod[['Product','TotalQty','TotalRevenue']].sort_values('TotalQty', ascending=False).to_string(index=False))


## 7. Customer Analysis

In [ ]:
cust = (master.groupby(['Customer','CustomerRegion'])
              .agg(Revenue=('Revenue','sum'),
                   Profit=('Profit','sum'),
                   Orders=('OrderID','count'))
              .reset_index()
              .sort_values('Revenue', ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors_c = sns.color_palette('husl', len(cust))

axes[0].barh(cust['Customer'], cust['Revenue'], color=colors_c, edgecolor='white')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'₹{v/1e6:.1f}M'))
axes[0].set_title('Revenue by Customer', fontsize=12, fontweight='bold')
axes[0].invert_yaxis()

axes[1].barh(cust['Customer'], cust['Profit'], color=colors_c, edgecolor='white')
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'₹{v/1e6:.1f}M'))
axes[1].set_title('Profit by Customer', fontsize=12, fontweight='bold')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('customer_analysis.png', bbox_inches='tight')
plt.show()

print("\nTop 5 Customers by Revenue:")
print(cust.head(5)[['Customer','CustomerRegion','Revenue','Profit','Orders']].to_string(index=False))


## 8. Delivery Performance Analysis

In [ ]:
# ── Delivery Status Pie + DeliveryDays Distribution ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

status_counts = master['DeliveryStatus'].value_counts()
colors_status = ['#4CAF50','#F44336','#2196F3']
axes[0].pie(status_counts, labels=status_counts.index, autopct='%1.1f%%',
            colors=colors_status, startangle=90,
            wedgeprops={'edgecolor':'white','linewidth':2})
axes[0].set_title('Delivery Status Distribution', fontsize=12, fontweight='bold')

axes[1].hist(master['DeliveryDays'], bins=7, color='#2196F3', edgecolor='white', alpha=0.85)
axes[1].axvline(master['DeliveryDays'].mean(), color='red', linestyle='--',
                linewidth=1.5, label=f"Mean: {master['DeliveryDays'].mean():.1f} days")
axes[1].set_title('Delivery Days Distribution', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Delivery Days')
axes[1].set_ylabel('Number of Orders')
axes[1].legend()

plt.tight_layout()
plt.savefig('delivery_analysis.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── Delayed Orders Analysis ───────────────────────────────────────────────────
delayed = master[master['DeliveryStatus'] == 'Delayed']

print(f"Total Delayed Orders: {len(delayed)}")
print(f"Revenue lost in delays: ₹{delayed['Revenue'].sum():,.0f}")
print(f"Profit lost in delays : ₹{delayed['Profit'].sum():,.0f}\n")

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Delays by product
d_prod = delayed.groupby('Product')['OrderID'].count().sort_values(ascending=False)
axes[0].bar(d_prod.index, d_prod.values, color='#F44336', edgecolor='white')
axes[0].set_title('Delayed Orders by Product', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Product')
axes[0].tick_params(axis='x', rotation=40)

# Delays by region
d_reg = delayed.groupby('CustomerRegion')['OrderID'].count().sort_values(ascending=False)
axes[1].bar(d_reg.index, d_reg.values, color='#FF9800', edgecolor='white')
axes[1].set_title('Delayed Orders by Region', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Region')

# Delays by driver
d_drv = delayed.groupby('Driver')['OrderID'].count().sort_values(ascending=False)
axes[2].barh(d_drv.index, d_drv.values, color='#9C27B0', edgecolor='white')
axes[2].set_title('Delayed Orders by Driver', fontsize=11, fontweight='bold')
axes[2].set_xlabel('Count')

plt.tight_layout()
plt.savefig('delayed_orders_breakdown.png', bbox_inches='tight')
plt.show()


## 9. Warehouse Performance

In [ ]:
wh = (master.groupby('Warehouse')
            .agg(Revenue=('Revenue','sum'),
                 Profit=('Profit','sum'),
                 Orders=('OrderID','count'),
                 AvgDelivery=('DeliveryDays','mean'),
                 DelayedCount=('DeliveryStatus', lambda x: (x=='Delayed').sum()))
            .reset_index()
            .sort_values('Revenue', ascending=False))

wh['DelayRate%'] = (wh['DelayedCount'] / wh['Orders'] * 100).round(1)

print("Warehouse Performance Summary:")
print(wh.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
pal_wh = sns.color_palette('Set2', len(wh))

axes[0].bar(wh['Warehouse'], wh['Revenue'], color=pal_wh, edgecolor='white')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'₹{v/1e6:.1f}M'))
axes[0].set_title('Revenue by Warehouse', fontsize=11, fontweight='bold')
axes[0].tick_params(axis='x', rotation=30)

axes[1].bar(wh['Warehouse'], wh['AvgDelivery'], color=pal_wh, edgecolor='white')
axes[1].set_title('Avg Delivery Days by Warehouse', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Days')
axes[1].tick_params(axis='x', rotation=30)

axes[2].bar(wh['Warehouse'], wh['DelayRate%'], color=pal_wh, edgecolor='white')
axes[2].set_title('Delay Rate % by Warehouse', fontsize=11, fontweight='bold')
axes[2].set_ylabel('Delay Rate (%)')
axes[2].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('warehouse_performance.png', bbox_inches='tight')
plt.show()


## 10. Driver Performance

In [ ]:
drv = (master.groupby(['Driver','Vehicle'])
             .agg(Orders=('OrderID','count'),
                  Revenue=('Revenue','sum'),
                  AvgDelivery=('DeliveryDays','mean'),
                  Delays=('DeliveryStatus', lambda x:(x=='Delayed').sum()))
             .reset_index()
             .sort_values('Revenue', ascending=False))

drv['DelayRate%'] = (drv['Delays'] / drv['Orders'] * 100).round(1)
print("Driver Performance:")
print(drv.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
pal_d = sns.color_palette('tab20', len(drv))

axes[0].barh(drv['Driver'], drv['Revenue'], color=pal_d, edgecolor='white')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'₹{v/1e6:.1f}M'))
axes[0].set_title('Revenue Delivered per Driver', fontsize=12, fontweight='bold')
axes[0].invert_yaxis()

axes[1].barh(drv['Driver'], drv['DelayRate%'], color=pal_d, edgecolor='white')
axes[1].set_title('Delay Rate % per Driver', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Delay Rate (%)')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('driver_performance.png', bbox_inches='tight')
plt.show()


## 11. Profitability & Loss Analysis

In [ ]:
# ── Negative profit orders ────────────────────────────────────────────────────
neg = master[master['Profit'] < 0].copy()
print(f"Total negative-profit orders: {len(neg)}")
print(f"Total loss amount: ₹{neg['Profit'].sum():,.0f}\n")
print("Loss Orders by Product:")
print(neg.groupby('Product')['Profit'].agg(['count','sum']).rename(columns={'count':'Orders','sum':'TotalLoss'}).to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Profit scatter
axes[0].scatter(master['Revenue'], master['Profit'],
                c=['#F44336' if p < 0 else '#4CAF50' for p in master['Profit']],
                alpha=0.6, s=40, edgecolors='white', linewidths=0.3)
axes[0].axhline(0, color='black', linestyle='--', linewidth=1)
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'₹{v/1e3:.0f}K'))
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'₹{v/1e3:.0f}K'))
axes[0].set_title('Revenue vs Profit (Red = Loss)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Revenue'); axes[0].set_ylabel('Profit')
legend_patches = [Patch(color='#4CAF50', label='Profit'), Patch(color='#F44336', label='Loss')]
axes[0].legend(handles=legend_patches)

# Profit distribution
axes[1].hist(master['Profit'], bins=40, color='#2196F3', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='red', linestyle='--', linewidth=1.5, label='Break-even')
axes[1].axvline(master['Profit'].mean(), color='orange', linestyle='--',
                linewidth=1.5, label=f"Mean: ₹{master['Profit'].mean():,.0f}")
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'₹{v/1e3:.0f}K'))
axes[1].set_title('Profit Distribution', fontsize=12, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('profitability_analysis.png', bbox_inches='tight')
plt.show()


## 12. Correlation Analysis

In [ ]:
num_cols = ['Quantity','Revenue','ShippingCost','TotalCost','Profit','DeliveryDays']
corr = master[num_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, ax=ax, square=True)
ax.set_title('Correlation Heatmap — Numeric Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', bbox_inches='tight')
plt.show()


## 13. 📋 Business Insights & Recommendations

### 🏆 KPI Summary
| KPI | Value |
|-----|-------|
| Total Revenue | ₹50M+ |
| Total Profit | ₹11M+ |
| Profit Margin | ~23% |
| Total Orders | 300 |
| On-Time Delivery Rate | ~77% |
| Delayed Orders | 35 (11.7%) |
| Negative Profit Orders | 41 |

---

### 🔍 Key Findings

**Products**
- 💻 **Laptop** is the highest-revenue product (₹55,000/unit × high qty)
- ⌨️ **Keyboard, Mouse, Router** frequently generate negative profit — pricing/cost review needed
- 🖨️ **Printer & Scanner** (Office category) show healthy margins

**Regions**
- 🌍 **South region** leads in both revenue and order volume
- 📉 **East region** has lowest contribution — potential growth opportunity

**Delivery**
- ⏱️ Average delivery time: **4 days** — within acceptable range
- 🚨 **35 orders delayed** — investigate warehouse/driver patterns
- 🏭 Warehouses with highest delay rates should be prioritized for operational review

**Customers**
- Top 3 customers contribute disproportionately to total revenue — dependency risk
- Customer retention strategies should focus on **BlueMart, Kingdom Mart, Lotus Hyper**

---

### 💡 Recommendations
1. **Review pricing** for Keyboard, Mouse, Router — 41 negative-profit orders is significant
2. **Investigate delayed orders** — identify if specific driver-warehouse combinations drive delays
3. **Expand South region** operations further (highest ROI region)
4. **Diversify customer base** — reduce dependency on top 3 customers
5. **Optimize ShippingCost** — it is weakly correlated with profit, suggesting room to negotiate
